# 05 Comparing missing-value strategies

For example, suppose a school is missing its teacher count. Notebook 05 tries three ways to predict its AP rate:

- Native XGBoost keeps the teacher count missing. Each tree learns which branch to follow when that value is missing.
- Imputation uses a random forest to estimate the missing teacher count from other school characteristics. XGBoost then predicts AP participation using the filled-in data.
- Reduced features uses a model trained on the features available for that school.


We use the same XGBoost settings, three random seeds, and one district-based validation split for all three strategies. We rank strategies by their mean validation RMSE. The standard deviation shows how much the score changes across seeds on these districts.

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import preprocessing_utils as pp

data = joblib.load("../data/processed/04_preprocessing.joblib")
features = data["feature_types"]
columns = [features[key] for key in ("cont_skewed", "cont_other", "cat_ftrs", "ord_ftrs")]
X_dev, y_dev, districts = data["X_other"], data["y_other"], data["groups_other"]
assert {"ap_rate", "enrl_AP", "enrl_9to12", "leaid"}.isdisjoint(X_dev.columns), "Rerun notebook 04"
split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=0)
train, valid = next(split.split(X_dev, groups=districts))
X_train, y_train = X_dev.iloc[train], y_dev.iloc[train]
assert set(districts.iloc[train]).isdisjoint(districts.iloc[valid])
X_valid, y_valid = X_dev.iloc[valid], y_dev.iloc[valid]

## Build the three pipelines

The helpers in `preprocessing_utils.py` log-transform skewed inputs, scale numerical features, and encode categories and grades. `make_pipeline` connects these steps to an AP-rate regressor. We fit each pipeline on the training districts.

Only the imputation pipeline uses a random forest to estimate missing inputs. It revisits each incomplete column using updated estimates from the previous steps. Shallow trees keep these updates simpler. The imputer stops when estimates settle or it reaches 12 rounds.

In [2]:
def make_pipeline(strategy, seed):
    model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, subsample=0.8,
                         colsample_bytree=0.8, random_state=seed, n_jobs=2)
    prep = pp.build_preprocessor(*columns)
    if strategy == "Imputation":
        prep = pp.build_imputation_preprocessor(*columns, seed=seed, n_estimators=10, max_iter=12)
    if strategy == "Reduced":
        model = pp.ReducedFeaturesRegressor(base_estimator=model)
    return Pipeline([("prep", prep), ("model", model)])

In [3]:
seeds = [0, 1, 2]
scores = []
for strategy in ["Native", "Imputation", "Reduced"]:
    for seed in seeds:
        model = make_pipeline(strategy, seed)
        model.fit(X_train, y_train)
        prediction = model.predict(X_valid)
        scores.append({"strategy": strategy, "seed": seed,
                       "rmse": root_mean_squared_error(y_valid, prediction)})

results = pd.DataFrame(scores)
print(results.to_string(index=False))
comparison = results.groupby("strategy").rmse.agg(["mean", "std"]).sort_values("mean")
print(comparison.round(5))
print(f"Lowest mean validation RMSE: {comparison.index[0]}")

  strategy  seed     rmse
    Native     0 0.138071
    Native     1 0.139497
    Native     2 0.132768
Imputation     0 0.137004
Imputation     1 0.135438
Imputation     2 0.138792
   Reduced     0 0.137557
   Reduced     1 0.138214
   Reduced     2 0.135123
               mean      std
strategy                    
Native      0.13678  0.00355
Reduced     0.13696  0.00163
Imputation  0.13708  0.00168
Lowest mean validation RMSE: Native


## Later model comparison

We will compare a training-mean baseline, a regularized linear model, random forest, XGBoost, and KNN. Models that require complete inputs can use the imputation pipeline. We will tune hyperparameters with five-fold GroupKFold and use RMSE to select the final model. We will also report MAE in AP participation percentage points.